# Exportação de dados para o dashboard web

Este notebook prepara um contrato de dados voltado à visualização, sem retreinar os modelos.

- **Entradas:** artefatos dos notebooks 03 e 05
- **Saídas tabulares:** Parquet para grids e predições
- **Saídas resumidas:** JSON com métricas, configurações, inventário e dados dos gráficos
- **Destino:** `src/db/outputs/08-export-web/`

> Execute os notebooks 03 e 05 antes deste. O navegador não acessa os arquivos Parquet diretamente; a API em `src/web/` fornece paginação, filtros e os dados dos gráficos em JSON.

## 1 - Setup e caminhos

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_ENTRADA = RAIZ / "src/db/outputs/05-avaliacao-metricas"
PASTA_PREPROCESSAMENTO = RAIZ / "src/db/outputs/03-preprocessamento"
PASTA_BASE = RAIZ / "src/db/outputs/08-export-web"
PASTA_ARTEFATOS = PASTA_BASE / "artifacts"
PASTA_METRICAS = PASTA_BASE / "metrics"

CAMINHO_MODELOS = PASTA_ENTRADA / "artifacts" / "modelos_treinados.joblib"
CAMINHO_METRICAS = PASTA_ENTRADA / "metrics" / "metricas_teste.json"
CAMINHO_PREPROCESSAMENTO = (
    PASTA_PREPROCESSAMENTO / "artifacts" / "dados_preprocessados.joblib"
)
CAMINHO_CSV = RAIZ / "src/db/models/breast-cancer-wisconsin-data/data.csv"

for pasta in (PASTA_ARTEFATOS, PASTA_METRICAS):
    pasta.mkdir(parents=True, exist_ok=True)

print(f"Entrada modelos: {CAMINHO_MODELOS}")
print(f"Entrada métricas: {CAMINHO_METRICAS}")
print(f"Entrada pré-processamento: {CAMINHO_PREPROCESSAMENTO}")
print(f"Saída web: {PASTA_BASE}")

Entrada modelos: E:\_git\fiap\tech-challenge-fase1-review\src\db\outputs\05-avaliacao-metricas\artifacts\modelos_treinados.joblib
Entrada métricas: E:\_git\fiap\tech-challenge-fase1-review\src\db\outputs\05-avaliacao-metricas\metrics\metricas_teste.json
Entrada pré-processamento: E:\_git\fiap\tech-challenge-fase1-review\src\db\outputs\03-preprocessamento\artifacts\dados_preprocessados.joblib
Saída web: E:\_git\fiap\tech-challenge-fase1-review\src\db\outputs\08-export-web


## 2 - Carregar modelos, amostras e métricas

A inferência é repetida apenas no conjunto de teste para gerar o contrato da interface. Nenhum modelo é treinado novamente.

In [2]:
if not CAMINHO_MODELOS.exists():
    raise FileNotFoundError("Execute o notebook 05 primeiro: falta modelos_treinados.joblib.")
if not CAMINHO_METRICAS.exists():
    raise FileNotFoundError("Execute o notebook 05 primeiro: falta metricas_teste.json.")
if not CAMINHO_PREPROCESSAMENTO.exists():
    raise FileNotFoundError("Execute o notebook 03 primeiro: faltam os dados preprocessados.")
if not CAMINHO_CSV.exists():
    raise FileNotFoundError(f"Dataset original não encontrado: {CAMINHO_CSV}")

pacote = joblib.load(CAMINHO_MODELOS)
pacote_preprocessamento = joblib.load(CAMINHO_PREPROCESSAMENTO)
dados_brutos = pd.read_csv(CAMINHO_CSV)
modelos_treinados = pacote["modelos_treinados"]
atributos_treino = pacote_preprocessamento["atributos_treino"]
rotulo_treino = pacote_preprocessamento["rotulo_treino"]
atributos_teste = pacote["atributos_teste"]
rotulo_teste = pacote["rotulo_teste"]
colunas_atributos = pacote["colunas_atributos"]
semente = int(pacote_preprocessamento["semente"])

with open(CAMINHO_METRICAS, encoding="utf-8") as arquivo:
    metricas_origem = json.load(arquivo)

nome_melhor = metricas_origem["_melhor_modelo"]
metricas_modelos = {
    nome: valores
    for nome, valores in metricas_origem.items()
    if not nome.startswith("_")
}

print(f"Modelos: {list(modelos_treinados)}")
print(f"Melhor modelo: {nome_melhor}")
print(f"Treino: {len(atributos_treino)} | Teste: {len(atributos_teste)}")
print(f"Atributos: {len(colunas_atributos)} | Semente: {semente}")

Modelos: ['Regressão Logística', 'Árvore de Decisão', 'SVM', 'Gradient Boosting', 'Random Forest']
Melhor modelo: SVM
Treino: 455 | Teste: 114
Atributos: 30 | Semente: 42


## 3 - Construir bases comparativas

- **Formato largo:** uma linha por amostra, ideal para o grid.
- **Formato longo:** uma linha por combinação amostra/modelo, ideal para gráficos e agregações.

In [3]:
rotulos = np.asarray(rotulo_teste, dtype=int)
dados_grid = atributos_teste.reset_index(drop=True).copy()
dados_grid.insert(0, "id", np.arange(1, len(dados_grid) + 1, dtype=int))
dados_grid["rotulo_real"] = rotulos
dados_grid["diagnostico_real"] = np.where(rotulos == 1, "Maligno", "Benigno")

linhas_longas = []
matrizes = {}
nomes_modelos = list(modelos_treinados)

for nome, pipeline in modelos_treinados.items():
    previstos = np.asarray(pipeline.predict(atributos_teste), dtype=int)
    probabilidades = np.asarray(pipeline.predict_proba(atributos_teste)[:, 1], dtype=float)
    acertos = previstos == rotulos

    dados_grid[f"pred_{nome}"] = previstos
    dados_grid[f"prob_{nome}"] = np.round(probabilidades, 6)
    dados_grid[f"acerto_{nome}"] = acertos

    matriz = confusion_matrix(rotulos, previstos, labels=[0, 1])
    tn, fp, fn, tp = (int(valor) for valor in matriz.ravel())
    matrizes[nome] = {
        "matriz": matriz.astype(int).tolist(),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

    for indice, (previsto, probabilidade, acerto) in enumerate(
        zip(previstos, probabilidades, acertos)
    ):
        linhas_longas.append(
            {
                "id": int(dados_grid.at[indice, "id"]),
                "modelo": nome,
                "rotulo_real": int(rotulos[indice]),
                "diagnostico_real": "Maligno" if rotulos[indice] == 1 else "Benigno",
                "rotulo_previsto": int(previsto),
                "diagnostico_previsto": "Maligno" if previsto == 1 else "Benigno",
                "prob_maligno": round(float(probabilidade), 6),
                "acerto": bool(acerto),
            }
        )

colunas_acerto = [f"acerto_{nome}" for nome in nomes_modelos]
colunas_probabilidade = [f"prob_{nome}" for nome in nomes_modelos]
dados_grid["n_modelos_acertaram"] = dados_grid[colunas_acerto].sum(axis=1).astype(int)
dados_grid["consenso"] = dados_grid["n_modelos_acertaram"].isin([0, len(nomes_modelos)])
dados_grid["amplitude_probabilidade"] = (
    dados_grid[colunas_probabilidade].max(axis=1)
    - dados_grid[colunas_probabilidade].min(axis=1)
).round(6)

dados_longos = pd.DataFrame(linhas_longas)

# Bases identificadas para os gráficos. O conjunto completo é a união exata do split.
atributos_completos = pd.concat([atributos_treino, atributos_teste], ignore_index=True)
rotulos_completos = pd.concat(
    [rotulo_treino.reset_index(drop=True), rotulo_teste.reset_index(drop=True)],
    ignore_index=True,
).astype(int)
conjuntos = {
    "completo": (atributos_completos, rotulos_completos),
    "treino": (atributos_treino.reset_index(drop=True), rotulo_treino.reset_index(drop=True).astype(int)),
    "teste": (atributos_teste.reset_index(drop=True), rotulo_teste.reset_index(drop=True).astype(int)),
}


def resumo_classes(rotulos_conjunto):
    total = int(len(rotulos_conjunto))
    contagens = rotulos_conjunto.value_counts().to_dict()
    return {
        "total": total,
        "benigno": {"n": int(contagens.get(0, 0)), "pct": float(contagens.get(0, 0) / total)},
        "maligno": {"n": int(contagens.get(1, 0)), "pct": float(contagens.get(1, 0) / total)},
    }


splits = {nome: resumo_classes(rotulos_split) for nome, (_, rotulos_split) in conjuntos.items()}

# Histogramas usam as mesmas faixas para as duas classes dentro de cada split.
histogramas = {}
for nome_split, (atributos_split, rotulos_split) in conjuntos.items():
    histogramas[nome_split] = {}
    for atributo in colunas_atributos:
        valores = atributos_split[atributo].astype(float).to_numpy()
        limites = np.histogram_bin_edges(valores, bins=16)
        centros = ((limites[:-1] + limites[1:]) / 2).round(6)
        histogramas[nome_split][atributo] = {
            "centros": centros.tolist(),
            "benigno": np.histogram(valores[rotulos_split.to_numpy() == 0], bins=limites)[0].astype(int).tolist(),
            "maligno": np.histogram(valores[rotulos_split.to_numpy() == 1], bins=limites)[0].astype(int).tolist(),
        }

base_correlacao = atributos_completos.copy()
base_correlacao["maligno"] = rotulos_completos.to_numpy()
matriz_correlacao = base_correlacao.corr().round(4)

# Importância global da árvore de decisão, calculada pelo classificador já treinado.
pipeline_arvore = modelos_treinados["Árvore de Decisão"]
classificador_arvore = pipeline_arvore.named_steps["classificador"]
importancias = sorted(
    [
        {"atributo": atributo, "valor": round(float(valor), 8)}
        for atributo, valor in zip(colunas_atributos, classificador_arvore.feature_importances_)
    ],
    key=lambda item: item["valor"],
    reverse=True,
)

print("Grid:", dados_grid.shape)
print("Predições longas:", dados_longos.shape)
print("Splits:", {nome: resumo["total"] for nome, resumo in splits.items()})
display(dados_grid.head(3))

Grid: (114, 51)
Predições longas: (570, 8)
Splits: {'completo': 569, 'treino': 455, 'teste': 114}


,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,acerto_SVM,pred_Gradient Boosting,prob_Gradient Boosting,acerto_Gradient Boosting,pred_Random Forest,prob_Random Forest,acerto_Random Forest,n_modelos_acertaram,consenso,amplitude_probabilidade
0,1,11.41,10.82,73.34,403.3,0.09373,0.06685,0.03512,0.02623,0.1667,...,True,0,0.000521,True,0,0.01,True,5,True,0.010000
1,2,20.94,23.56,138.90,1364.0,0.10070,0.16060,0.27120,0.13100,0.2205,...,True,1,0.999557,True,1,1.00,True,5,True,0.001737
2,3,16.17,16.07,106.30,788.5,0.09880,0.14380,0.06651,0.05397,0.1990,...,True,0,0.220210,True,0,0.34,True,4,False,0.957432


## 4 - Montar métricas, rankings e metadados

Empates são preservados: mais de um modelo pode aparecer como melhor ou pior na mesma métrica.

In [4]:
METRICAS = ["acuracia", "recall_maligno", "f1_maligno", "precisao_maligno"]

ranking = {}
for metrica in METRICAS:
    ordenados = sorted(
        metricas_modelos.items(),
        key=lambda item: float(item[1][metrica]),
        reverse=True,
    )
    ranking[metrica] = [
        {
            "modelo": nome,
            "valor": float(valores[metrica]),
            "posicao": posicao,
        }
        for posicao, (nome, valores) in enumerate(ordenados, start=1)
    ]

classificadores = {
    nome: pipeline.named_steps["classificador"]
    for nome, pipeline in modelos_treinados.items()
}
parametros = {nome: classificador.get_params() for nome, classificador in classificadores.items()}

modelos_configuracao = {
    "Regressão Logística": {
        "algoritmo": type(classificadores["Regressão Logística"]).__name__,
        "justificativa": "Modelo linear probabilístico, simples de interpretar e adequado à triagem.",
        "hiperparametros": {
            "max_iter": int(parametros["Regressão Logística"]["max_iter"]),
            "random_state": int(parametros["Regressão Logística"]["random_state"]),
        },
    },
    "Árvore de Decisão": {
        "algoritmo": type(classificadores["Árvore de Decisão"]).__name__,
        "justificativa": "Captura relações não lineares e fornece importância global dos atributos.",
        "hiperparametros": {
            "random_state": int(parametros["Árvore de Decisão"]["random_state"]),
        },
    },
    "SVM": {
        "algoritmo": "SVC calibrado",
        "justificativa": "Margem não linear com balanceamento de classes e probabilidades calibradas.",
        "hiperparametros": {
            "kernel": parametros["SVM"]["estimator__kernel"],
            "class_weight": parametros["SVM"]["estimator__class_weight"],
            "calibracao": parametros["SVM"]["method"],
            "cv": int(parametros["SVM"]["cv"]),
            "ensemble": bool(parametros["SVM"]["ensemble"]),
        },
    },
    "Gradient Boosting": {
        "algoritmo": type(classificadores["Gradient Boosting"]).__name__,
        "justificativa": "Combina árvores sequenciais para corrigir erros e capturar relações complexas.",
        "hiperparametros": {
            "random_state": int(parametros["Gradient Boosting"]["random_state"]),
        },
    },
    "Random Forest": {
        "algoritmo": type(classificadores["Random Forest"]).__name__,
        "justificativa": "Combina muitas árvores para reduzir variância e usa balanceamento de classes.",
        "hiperparametros": {
            "n_estimators": int(parametros["Random Forest"]["n_estimators"]),
            "class_weight": parametros["Random Forest"]["class_weight"],
            "random_state": int(parametros["Random Forest"]["random_state"]),
            "n_jobs": int(parametros["Random Forest"]["n_jobs"]),
        },
    },
}

preprocessamento = {
    "imputacao": "mediana",
    "padronizacao": "StandardScaler",
    "ajuste": "somente no conjunto de treino",
    "atributos": len(colunas_atributos),
}

por_modelo = {}
for nome, valores in metricas_modelos.items():
    melhores = {
        metrica
        for metrica in METRICAS
        if np.isclose(
            float(valores[metrica]),
            max(float(item[metrica]) for item in metricas_modelos.values()),
        )
    }
    piores = {
        metrica
        for metrica in METRICAS
        if np.isclose(
            float(valores[metrica]),
            min(float(item[metrica]) for item in metricas_modelos.values()),
        )
    }
    por_modelo[nome] = {
        **{metrica: float(valores[metrica]) for metrica in METRICAS},
        **matrizes[nome],
        "melhor_em": sorted(melhores),
        "pior_em": sorted(piores),
        "configuracao": {
            **modelos_configuracao[nome],
            "amostras_treino": len(atributos_treino),
            "amostras_avaliacao": len(atributos_teste),
        },
    }

payload_metricas = {
    "versao_contrato": 2,
    "criterio_selecao": "recall_maligno",
    "melhor_modelo": nome_melhor,
    "quantidade_modelos": len(nomes_modelos),
    "quantidade_amostras": len(dados_grid),
    "quantidade_atributos": len(colunas_atributos),
    "atributos": list(colunas_atributos),
    "preprocessamento": preprocessamento,
    "ranking": ranking,
    "por_modelo": por_modelo,
}

payload_graficos = {
    "versao_contrato": 2,
    "inventario": {
        "registros_brutos": len(dados_brutos),
        "registros_usados": len(atributos_completos),
        "registros_descartados": len(dados_brutos) - len(atributos_completos),
        "registros_duplicados_origem": int(dados_brutos.duplicated().sum()),
        "valores_ausentes_origem": int(dados_brutos.isna().sum().sum()),
        "colunas_removidas_modelagem": ["id", "Unnamed: 32"],
        "quantidade_atributos": len(colunas_atributos),
        "semente": semente,
        "test_size": 0.2,
        "estratificado": True,
    },
    "splits": splits,
    "histogramas": histogramas,
    "correlacao": {
        "atributos": matriz_correlacao.columns.tolist(),
        "matriz": matriz_correlacao.to_numpy().tolist(),
        "conjunto": "completo",
    },
    "importancia_atributos": {
        "modelo": "Árvore de Decisão",
        "metodo": "feature_importances_ (redução média de impureza)",
        "itens": importancias,
    },
}

print(json.dumps({
    "melhor_modelo": payload_metricas["melhor_modelo"],
    "quantidade_modelos": payload_metricas["quantidade_modelos"],
    "quantidade_amostras_teste": payload_metricas["quantidade_amostras"],
    "quantidade_amostras_treino": splits["treino"]["total"],
}, ensure_ascii=False, indent=2))

{
  "melhor_modelo": "SVM",
  "quantidade_modelos": 5,
  "quantidade_amostras_teste": 114,
  "quantidade_amostras_treino": 455
}


## 5 - Exportar Parquet e JSON

In [5]:
CAMINHO_GRID = PASTA_ARTEFATOS / "amostras_comparativo.parquet"
CAMINHO_LONGO = PASTA_ARTEFATOS / "predicoes_longas.parquet"
CAMINHO_METRICAS_WEB = PASTA_METRICAS / "metricas_comparativo.json"
CAMINHO_GRAFICOS_WEB = PASTA_METRICAS / "graficos.json"

dados_grid.to_parquet(CAMINHO_GRID, index=False, engine="pyarrow", compression="snappy")
dados_longos.to_parquet(CAMINHO_LONGO, index=False, engine="pyarrow", compression="snappy")

with open(CAMINHO_METRICAS_WEB, "w", encoding="utf-8") as arquivo:
    json.dump(payload_metricas, arquivo, indent=2, ensure_ascii=False)
with open(CAMINHO_GRAFICOS_WEB, "w", encoding="utf-8") as arquivo:
    json.dump(payload_graficos, arquivo, indent=2, ensure_ascii=False)

print(f"Grid: {CAMINHO_GRID}")
print(f"Predições: {CAMINHO_LONGO}")
print(f"Métricas: {CAMINHO_METRICAS_WEB}")
print(f"Gráficos: {CAMINHO_GRAFICOS_WEB}")

Grid: E:\_git\fiap\tech-challenge-fase1-review\src\db\outputs\08-export-web\artifacts\amostras_comparativo.parquet
Predições: E:\_git\fiap\tech-challenge-fase1-review\src\db\outputs\08-export-web\artifacts\predicoes_longas.parquet
Métricas: E:\_git\fiap\tech-challenge-fase1-review\src\db\outputs\08-export-web\metrics\metricas_comparativo.json
Gráficos: E:\_git\fiap\tech-challenge-fase1-review\src\db\outputs\08-export-web\metrics\graficos.json


## 6 - Validar o contrato exportado

In [6]:
assert CAMINHO_GRID.exists()
assert CAMINHO_LONGO.exists()
assert CAMINHO_METRICAS_WEB.exists()
assert CAMINHO_GRAFICOS_WEB.exists()

validacao_grid = pd.read_parquet(CAMINHO_GRID)
validacao_longo = pd.read_parquet(CAMINHO_LONGO)
with open(CAMINHO_METRICAS_WEB, encoding="utf-8") as arquivo:
    validacao_metricas = json.load(arquivo)
with open(CAMINHO_GRAFICOS_WEB, encoding="utf-8") as arquivo:
    validacao_graficos = json.load(arquivo)

assert len(validacao_grid) == len(atributos_teste)
assert len(validacao_longo) == len(atributos_teste) * len(modelos_treinados)
assert validacao_grid["id"].is_unique
assert set(validacao_longo["modelo"]) == set(modelos_treinados)
assert validacao_metricas["melhor_modelo"] in modelos_treinados
assert all(f"prob_{nome}" in validacao_grid for nome in modelos_treinados)
assert all(f"acerto_{nome}" in validacao_grid for nome in modelos_treinados)
assert validacao_graficos["splits"]["treino"]["total"] == len(atributos_treino)
assert validacao_graficos["splits"]["teste"]["total"] == len(atributos_teste)
assert len(validacao_graficos["correlacao"]["matriz"]) == len(colunas_atributos) + 1
assert len(validacao_graficos["importancia_atributos"]["itens"]) == len(colunas_atributos)
assert all("configuracao" in validacao_metricas["por_modelo"][nome] for nome in modelos_treinados)

print("Contrato web validado com sucesso.")
print(f"{len(validacao_grid)} amostras × {len(modelos_treinados)} modelos")
print(f"{validacao_graficos['splits']['treino']['total']} treino + {validacao_graficos['splits']['teste']['total']} teste")

Contrato web validado com sucesso.
114 amostras × 5 modelos
455 treino + 114 teste
